In [1]:
import os

from ase.data.pubchem import pubchem_atoms_search
from ase.visualize import view
import numpy as np
from ase import Atoms
from ase.build import molecule
# import mace off
from mace.calculators import mace_off
from ase.optimize import BFGS

from ase.io import read, write


import nqetools as nqe



/home/louie/anaconda3/envs/ipi_env/lib/python3.12/site-packages/e3nn/o3/_wigner.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  _Jd, _W3j_flat, _W3j_indices = torch.loa

In [2]:
def add_hydrogen_halfway(atoms, index1, index2):
    """
    Add a hydrogen atom halfway between two atoms in an Atoms object.

    Parameters:
    atoms (Atoms): The ASE Atoms object.
    index1 (int): The index of the first atom.
    index2 (int): The index of the second atom.

    Returns:
    Atoms: The updated Atoms object with the hydrogen atom added.
    """
    atoms = atoms.copy()
    # Get the positions of the two atoms
    pos1 = atoms.positions[index1]
    pos2 = atoms.positions[index2]

    # Calculate the midpoint
    midpoint = (pos1 + pos2) / 2.0

    # Add a hydrogen atom at the midpoint
    atoms += Atoms('H', positions=[midpoint])

    return atoms

def add_hydrogen_at_distance(atoms, index1, index2, distance):
    """
    Add a hydrogen atom at a specified distance from one atom along the line between two atoms in an Atoms object.

    Parameters:
    atoms (Atoms): The ASE Atoms object.
    index1 (int): The index of the first atom.
    index2 (int): The index of the second atom.
    distance (float): The distance from the first atom to place the hydrogen atom.

    Returns:
    Atoms: The updated Atoms object with the hydrogen atom added.
    """
    atoms = atoms.copy()
    # Get the positions of the two atoms
    pos1 = atoms.positions[index1]
    pos2 = atoms.positions[index2]

    # Calculate the direction vector from atom1 to atom2
    direction = pos2 - pos1
    direction /= np.linalg.norm(direction)  # Normalize the direction vector

    # Calculate the position of the hydrogen atom
    hydrogen_position = pos1 + direction * distance

    # Add a hydrogen atom at the calculated position
    atoms += Atoms('H', positions=[hydrogen_position])

    return atoms


def swap_bonding_configuration(atoms, donor_index, hydrogen_index, acceptor_index):
    """
    Swap the bonding configuration from O-H...O to O...H-O in an Atoms object.

    Parameters:
    atoms (Atoms): The ASE Atoms object.
    donor_index (int): The index of the donor oxygen atom.
    hydrogen_index (int): The index of the hydrogen atom.
    acceptor_index (int): The index of the acceptor oxygen atom.

    Returns:
    Atoms: The updated Atoms object with the swapped bonding configuration.
    """
    # Get the positions of the donor, hydrogen, and acceptor atoms
    donor_pos = atoms.positions[donor_index]
    hydrogen_pos = atoms.positions[hydrogen_index]
    acceptor_pos = atoms.positions[acceptor_index]

    # Calculate the new position for the hydrogen atom
    direction = acceptor_pos - donor_pos
    direction /= np.linalg.norm(direction)  # Normalize the direction vector
    new_hydrogen_pos = acceptor_pos - direction * np.linalg.norm(hydrogen_pos - donor_pos)

    # Update the position of the hydrogen atom
    atoms.positions[hydrogen_index] = new_hydrogen_pos

    return atoms

In [3]:
fmax = 0.01
calc = mace_off(model="large")

Using MACE-OFF23 MODEL for MACECalculator with /home/louie/.cache/mace/MACE-OFF23_large.model
Using float64 for MACECalculator, which is slower but more accurate. Recommended for geometry optimization.


/home/louie/anaconda3/envs/ipi_env/lib/python3.12/site-packages/mace/calculators/mace.py:128: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(f=model_path, map_loca

In [4]:
atoms = pubchem_atoms_search(smiles="C(C=O)C=O")
reactant = add_hydrogen_at_distance(atoms, 0, 1, 1.0)
product = add_hydrogen_at_distance(atoms, 1, 0, 1.0)
# view(reactant)
# view(product)

/home/louie/anaconda3/envs/ipi_env/lib/python3.12/site-packages/ase/data/pubchem.py:80: UserWarning: The structure "C(C=O)C=O" has more than one conformer in PubChem. By default, the first conformer is returned, please ensure you are using the structure you intend to or use the `ase.data.pubchem.pubchem_conformer_search` function
  warnings.warn(


In [5]:
reactant = nqe.optimise_geom(reactant, calc, fmax=fmax)
view(reactant)

Optimizing geometry...
      Step     Time          Energy          fmax
BFGS:    0 16:29:33    -7288.912628        4.994655
BFGS:    1 16:29:34    -7289.172125        2.716371
BFGS:    2 16:29:35    -7289.393162        2.222009
BFGS:    3 16:29:35    -7289.571991        2.609442
BFGS:    4 16:29:35    -7289.679488        1.069424
BFGS:    5 16:29:35    -7289.707226        0.431133
BFGS:    6 16:29:36    -7289.741663        0.483204
BFGS:    7 16:29:36    -7289.769846        0.455503
BFGS:    8 16:29:36    -7289.776675        0.224573
BFGS:    9 16:29:36    -7289.780693        0.172483
BFGS:   10 16:29:36    -7289.783577        0.182813
BFGS:   11 16:29:37    -7289.785358        0.143568
BFGS:   12 16:29:37    -7289.786643        0.099709
BFGS:   13 16:29:37    -7289.787976        0.137749
BFGS:   14 16:29:37    -7289.789442        0.158731
BFGS:   15 16:29:38    -7289.790592        0.117974
BFGS:   16 16:29:38    -7289.791233        0.055960
BFGS:   17 16:29:38    -7289.791532        

<Popen: returncode: None args: ['/home/louie/anaconda3/envs/ipi_env/bin/pyth...>

In [6]:
product = nqe.optimise_geom(product, calc, fmax=fmax)
view(product)

Optimizing geometry...
      Step     Time          Energy          fmax
BFGS:    0 16:29:40    -7288.912767        4.993377
BFGS:    1 16:29:40    -7289.172130        2.716493
BFGS:    2 16:29:40    -7289.393168        2.222044
BFGS:    3 16:29:41    -7289.572215        2.609294
BFGS:    4 16:29:41    -7289.679472        1.070681
BFGS:    5 16:29:41    -7289.707230        0.431148
BFGS:    6 16:29:41    -7289.741633        0.483269
BFGS:    7 16:29:42    -7289.769850        0.456667
BFGS:    8 16:29:42    -7289.776677        0.224487
BFGS:    9 16:29:42    -7289.780694        0.172507
BFGS:   10 16:29:42    -7289.783579        0.183300
BFGS:   11 16:29:43    -7289.785359        0.143909
BFGS:   12 16:29:43    -7289.786643        0.099642
BFGS:   13 16:29:43    -7289.787977        0.137907
BFGS:   14 16:29:43    -7289.789443        0.158824
BFGS:   15 16:29:44    -7289.790592        0.118135
BFGS:   16 16:29:44    -7289.791233        0.055948
BFGS:   17 16:29:44    -7289.791532        

<Popen: returncode: None args: ['/home/louie/anaconda3/envs/ipi_env/bin/pyth...>

In [8]:
neb = nqe.prepare_neb(reactant, product, calc, n_images=5)
view(neb.images)

<Popen: returncode: None args: ['/home/louie/anaconda3/envs/ipi_env/bin/pyth...>

In [10]:
ts_path = nqe.optimise_neb(neb, fmax=fmax)

      Step     Time          Energy          fmax
BFGS:    0 16:32:55    -7289.790732        0.009055


In [11]:
view(ts_path)

<Popen: returncode: None args: ['/home/louie/anaconda3/envs/ipi_env/bin/pyth...>

In [13]:
def get_ts_image(neb_images):
    # Find the image with the highest energy
    index = np.argmax([image.calc.get_potential_energy() for image in neb_images])
    return neb_images[index]
ts_image = get_ts_image(ts_path)
view(ts_image)

PropertyNotImplementedError: The property "energy" is not available.